<p><font size="6" color='grey'> <b>
Machine Learning
</b></font> </br></p>


<p><font size="5" color='grey'> <b>
Bootstrapping - Decision Tree - Titanic
</b></font> </br></p>

---


# 0  | Install & Import
***

In [ ]:
# Install

In [ ]:
# Daten & Strukturen
from pandas import read_excel  # Excel laden

# Numerische Operationen
from numpy.random import choice  # Zufallsstichproben ziehen
from numpy import array, percentile, mean  # Numerische Operationen

# Datenvorverarbeitung
from sklearn.impute import SimpleImputer  # Fehlende Werte imputieren
from sklearn.preprocessing import OrdinalEncoder, MinMaxScaler  # Encoding und Skalierung

# Datenaufteilung
from sklearn.model_selection import train_test_split  # Train/Test-Split

# Modell
from sklearn.tree import DecisionTreeClassifier  # Klassifikationsmodell

# Evaluation & Metriken
from sklearn.metrics import (
    accuracy_score,           # Genauigkeit
    confusion_matrix,         # Konfusionsmatrix
    ConfusionMatrixDisplay,   # Visualisierung der Matrix
    classification_report,    # Detaillierter Bericht
)

# Visualisierung
import plotly.express as px  # Einfache Plots

In [ ]:
# Warnung ausstellen
import warnings
warnings.filterwarnings("ignore")

# 1  | Understand
***


<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Aufgabe verstehen</br>
✅ Daten sammeln</br>
✅ Statistische Analyse (Min, Max, Mean, Korrelation, ...)</br>
✅ Datenvisualisierung (Streudiagramm, Box-Plot, ...)</br>
✅ Prepare Schritte festlegen</br>

<p><font color='black' size="5">
📒 Anwendungsfall
</font></p>

Dies ist der legendäre Titanic ML-Wettbewerb – die beste erste Herausforderung, um in ML-Modellierung einzutauchen.

Die Aufgabe ist einfach: Mit maschinellem Lernen lässt sich ein Modell erstellen, das vorhersagt, welche Passagiere den Schiffbruch der Titanic überlebt haben.

[Titanic Org](https://www.encyclopedia-titanica.org/)

[DataSet](https://www.openml.org/search?type=data&status=active&id=41265)

[Info](https://www.kaggle.com/competitions/titanic/data)


**Datenfelder:**   
+ Age: Alter
+ Fare: Ticketpreis
+ Sex: Geschlecht (0 = männlich, 1 = weiblich)
+ sibsp: Der Datensatz definiert Familienbeziehungen auf diese Weise ... Geschwister = Bruder, Schwester, Stiefbruder, Stiefschwester Ehepartner = Ehemann, Ehefrau (Geliebte und Verlobte wurden ignoriert)
+ parch: Der Datensatz definiert Familienbeziehungen auf diese Weise ... Elternteil = Mutter, Vater Kind = Tochter, Sohn, Stieftochter, Stiefsohn. Einige Kinder reisten nur mit einem Kindermädchen, daher ist für sie Parch=0
+ Pclass: Passagierklasse, 1.- 3. Klasse
+ Embarked: Hafen der Einschiffung

In [ ]:
df = read_excel(
    "https://raw.githubusercontent.com/ralf-42/ML_Intro/main/02_daten/05_tabellen/titanic.xlsx",
    usecols=["pclass", "survived", "sex", "age", "sibsp", "parch"],
)

In [ ]:
data = df.copy()
target = data.pop("survived")

<p><font color='black' size="5">
EDA (Exploratory Data Analysis) mit Pandas
</font></p>

In [ ]:
data.info()

In [ ]:
data.describe().T

In [ ]:
data.groupby("sex").count()

In [ ]:
target.value_counts()

#  2 | Prepare

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Train-Test-Split durchführen</br>
✅ Nicht benötigte Features löschen</br>
✅ Datentyp ermitteln/ändern</br>
✅ Missing Values behandeln</br>
✅ Ausreißer behandeln</br>
✅ Kategorischer Features Kodieren</br>
✅ Numerischer Features skalieren</br>
✅ Feature-Engineering (neue Features schaffen)</br>
✅ Dimensionalität reduzieren</br>
✅ Resampling (Over-/Undersampling)</br>
✅ Pipeline erstellen/konfigurieren</br>


<font color='black' size="5">
Datentyp ermitteln
</font>

In [ ]:
all_col = data.columns
num_col = data.select_dtypes(include="number").columns
cat_col = data.select_dtypes(exclude="number").columns

<font color='black' size="5">
Train-Test-Split
</font>

In [ ]:
data_train, data_test, target_train, target_test = train_test_split(
    data, target, test_size=0.20, stratify=target, random_state=42)

<font color='black' size="5">
Kodierung
</font>

In [ ]:
encoder = OrdinalEncoder()
encoder.fit(data_train[cat_col])
data_train[cat_col] = encoder.transform(data_train[cat_col])
data_test[cat_col] = encoder.transform(data_test[cat_col])

<font color='black' size="5">
Skalierung
</font>

In [ ]:
scaler = MinMaxScaler()
scaler.fit(data_train[all_col])
data_train[all_col] = scaler.transform(data_train[all_col])
data_test[all_col] = scaler.transform(data_test[all_col])

# 3 | Modeling
---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Modellauswahl treffen</br>
✅ Pipeline erweitern/konfigurieren</br>
✅ Training durchführen</br>
✅ Hyperparameter Tuning</br>
✅ Cross-Validiation</br>
✅ Bootstrapping</br>
✅ Regularization</br>

<p><font color='black' size="5">🌳 Algorithmus: Entscheidungsbaum (Decision Tree – Klassifikation)</font></p>

Ein Entscheidungsbaum trifft Entscheidungen wie ein Ablaufdiagramm: Ausgehend von einer Wurzel stellt er Ja/Nein-Fragen zu den Merkmalen und verzweigt je nach Antwort – bis er eine Vorhersage (Blatt) erreicht.

**Analogie:** Beim Ratespiel 'Wer bin ich?' werden nacheinander Fragen gestellt – am Ende steht die Antwort. Genau so arbeitet ein Entscheidungsbaum.

**Wichtige Parameter:**

| Parameter | Bedeutung |
|-----------|----------|
| `max_depth` | Maximale Tiefe (begrenzt Overfitting) |
| `criterion` | Gütekriterium für Splits: `gini` oder `entropy` |
| `random_state` | Reproduzierbarkeit |

**Wann Bootstrapping?**

**In der Praxis relevant wenn:**
- Die Unsicherheit einer Modellgüteschätzung quantifiziert werden soll (Konfidenzintervall für Accuracy)
- Der Datensatz zu klein ist für stabile Kreuzvalidierung
- Robuste statistische Aussagen ohne Normalverteilungsannahme benötigt werden

**Nicht geeignet wenn:**
- Rechenzeit kritisch ist → Bootstrapping mit 1.000 Iterationen ist teurer als 5-fache Kreuzvalidierung
- Eine schnelle Ersteinschätzung gefragt ist → dann reicht ein einfacher Train-Test-Split

**Typischer Fehler:**
Zu wenige Bootstrap-Samples verwenden (< 200). Erst ab 1.000 Samples stabilisieren sich Konfidenzintervalle zuverlässig — mit weniger Samples schwanken die Intervallgrenzen stark.

In [ ]:
#@markdown   <p><font size="4" color='green'>  Entscheidungsbaum</font> </br></p>

import base64
from IPython.display import Image, display

diagram = """
flowchart TD
    ROOT[Wurzel: Weiblich?] -->|ja| A[Klasse 1 oder 2?]
    ROOT -->|nein| B[Alter < 16?]

    A -->|ja| C[Überlebensrate: 94.2%]
    A -->|nein| D[Überlebensrate: 50.0%]

    B -->|ja| E[Überlebensrate: 38.2%]
    B -->|nein| F[Überlebensrate: 16.5%]
"""

encoded = base64.urlsafe_b64encode(diagram.strip().encode()).decode()
display(Image(url=f"https://mermaid.ink/img/{encoded}", width=700))

<p><font color='black' size="5">
Modellauswahl & Training
</font></p>

In [ ]:
model = DecisionTreeClassifier(max_depth=3, random_state=42)

In [ ]:
model.fit(data_train, target_train)

# 4 | Evaluate
---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Prognose (Train, Test) erstellen</br>
✅ Modellgüte prüfen</br>
✅ Residuenanalyse erstellen</br>
✅ Feature Importance/Selektion prüfen</br>
✅ Robustheitstest erstellen</br>
✅ Modellinterpretation erstellen</br>
✅ Sensitivitätsanalyse erstellen</br>
✅ Kommunikation (Key Takeaways)</br>

<p><font color='black' size="5">
Prognose
</font></p>

In [ ]:
target_train_pred = model.predict(data_train)
target_test_pred = model.predict(data_test)

<p><font color='black' size="5">
Confusion Matrix
</font></p>

In [ ]:
conf_matrix = confusion_matrix(target_test, target_test_pred)
display_labels_ = ["Not Survived", "Survived"]
disp = ConfusionMatrixDisplay(conf_matrix, display_labels=display_labels_)
disp.plot(cmap="Blues")

In [ ]:
print(classification_report(target_test, target_test_pred, target_names=display_labels_))

<p><font color='black' size="5">
Accuracy
</font></p>

In [ ]:
acc_train = accuracy_score(target_train, target_train_pred) * 100
print(f"Modell: {model} -- Train -- Accuracy: {acc_train:5.2f}")

In [ ]:
acc_test = accuracy_score(target_test, target_test_pred) * 100
print(f"Modell: {model} -- Test -- Accuracy: {acc_test:5.2f}%")

<p><font size="5">
Feature Importance
</p>

In [ ]:
title_ = "Feature Importance Titanic"
px.bar(
    x=model.feature_importances_, y=data.columns, title=title_, width=800, height=600
).update_yaxes(categoryorder="total ascending")

<p><font size="5">
Bootstrapping
</p>



Bootstrapping ist in der Statistik eine Resampling-Methode, bei der aus einer vorhandenen Stichprobe wiederholt neue Stichproben gleicher Größe mit Zurücklegen gezogen werden.

Für jede dieser sogenannten Bootstrap-Stichproben wird die gewünschte Statistik (z.B. Mittelwert, Median, Varianz) berechnet. Die so entstehende empirische Verteilung dieser Werte erlaubt es, Konfidenzintervalle, Standardfehler oder die Präzision von Schätzungen zu bestimmen - auch wenn die zugrunde liegende Verteilung unbekannt ist oder klassische Annahmen wie Normalverteilung nicht erfüllt werden können.

In [ ]:
n_bootstrap_samples = 1000  # Empfehlung ist 1000!
bootstrap_accuracy = []

In [ ]:
# Schleife für Anzahl der Bootstrap-Samples
for j in range(n_bootstrap_samples):
    # Bootstrap-Stichprobe mit Zurücklegen ziehen
    bootstrap_indices = choice(len(data_train), size=len(data_train), replace=True)

    # Effiziente OOB-Ermittlung über Set-Operationen
    bootstrap_set = set(bootstrap_indices)
    oob_indices = array([i for i in range(len(data_train)) if i not in bootstrap_set])

    # Bootstrap- und OOB-Daten erstellen
    data_bootstrap = data_train.iloc[bootstrap_indices]  # iloc ist sicherer als loc
    target_bootstrap = target_train.iloc[bootstrap_indices]
    data_oob = data_train.iloc[oob_indices]
    target_oob = target_train.iloc[oob_indices]

    # Modell trainieren und bewerten
    model.fit(data_bootstrap, target_bootstrap)
    target_oob_pred = model.predict(data_oob)
    accuracy = accuracy_score(target_oob, target_oob_pred)
    bootstrap_accuracy.append(accuracy)

    # Fortschrittsanzeige
    if j % 100 == 0:
        print(f"Sample: {j:>7,.0f}   Train: {len(bootstrap_indices):>7,.0f}    OOB: {len(oob_indices):>7,.0f}")

<p><font color='black' size="5">
Konfidenzintervall
</font></p>

Ein Konfidenzintervall ist ein Bereich, der angibt, in welchem Wertebereich ein unbekannter Parameter (z. B. Mittelwert, Genauigkeit, ...) mit einer bestimmten Wahrscheinlichkeit liegt.

**Einfach:**    
Ein Konfidenzintervall sagt: "Wir sind ziemlich sicher, dass der wahre Wert irgendwo in diesem Bereich liegt."

In [ ]:
# Berechnung des mittleren Accuracy und des Konfidenzintervalls
mean_accuracy = mean(bootstrap_accuracy)
conf_interval_80 = percentile(bootstrap_accuracy, [10, 90])
conf_interval_95 = percentile(bootstrap_accuracy, [2.5, 97.5])
conf_interval_99 = percentile(bootstrap_accuracy, [0.5, 99.5])

In [ ]:
print(f"Mittlerer Accuracy: {mean_accuracy:.4f}")
print("Konfidenzintervall Accuracy:")
print(f"80% KI/CI: [{conf_interval_80[0]:.4f},  {conf_interval_80[1]:.4f}]")
print(f"95% KI/CI: [{conf_interval_95[0]:.4f},  {conf_interval_95[1]:.4f}]")
print(f"99% KI/CI: [{conf_interval_99[0]:.4f},  {conf_interval_99[1]:.4f}]")

In [ ]:
fig = px.histogram(
    bootstrap_accuracy,
    nbins=50,
    title="Verteilung der Bootstrap Accuracy (95% Konfidenzintervall)",
    labels={"value": "Accuracy"},
    width=1000, # Breite der Grafik
    height=600, # Höhe der Grafik
)
fig.add_vline(
    x=conf_interval_95[0],
    line_dash="dash",
    line_color="red",
    annotation_text=f"95% CI Lower: {conf_interval_95[0]:.4f}",
    annotation_position="top left",
)
fig.add_vline(
    x=conf_interval_95[1],
    line_dash="dash",
    line_color="red",
    annotation_text=f"95% CI Upper: {conf_interval_95[1]:.4f}",
    annotation_position="top right",
)
fig.show()

# 5 | Deploy
---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Modellexport und -speicherung</br>
✅ Abhängigkeiten und Umgebung</br>
✅ Sicherheit und Datenschutz</br>
✅ In die Produktion integrieren</br>
✅ Tests und Validierung</br>
✅ Dokumentation & Wartung</br>